# Naive RAG

**[Knowledge base building process]**  
- Step1. Data Load(Document Loader)  
- Step2. Chunking(Text Splitter)  
- Step3. Vectorizing(Embedding)  
- Step4. Vector DB storage  

**[Runtime process]**  
- Step1. Query vectorization  
- Step2. Retrieval  
- Step3. LLM request (RAG Chain 구성)  
- Step4. 답변 출력  

[Data]  
- 저작권 판례 Data set

[Note]
- Advanced RAG Techniques를 더해 응답 품질을 개선함
- KB는 판례 + QA(평가 holdout 제외)로 구성
- 평가에는 QA holdout만 사용하므로, 평가 질문 원문 자체는 KB에 들어가지 않음
- retrieval 단계에서는 QA 문서에 보너스를 주고, 최종 context는 `판례 3개 + QA 2개` 우선 혼합으로 구성
- 목표: `factual_correctness` 를 높이면서 판례 근거를 함께 유지

In [1]:
%pip install -q langchain langchain-openai langchain-community langchain-chroma chromadb ragas


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

# Open AI API Key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")


In [3]:
# Local workspace path
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
DATASET_PATH = PROJECT_ROOT / "저작권판례데이터셋"
CHROMA_PATH = PROJECT_ROOT / "chroma_db"
CHROMA_PATH.mkdir(exist_ok=True)

print("현재 작업 경로:", PROJECT_ROOT)
print("데이터셋 경로:", DATASET_PATH)
print("ChromaDB 저장 경로:", CHROMA_PATH)

현재 작업 경로: /Users/hjcha/Documents/NavieRAG실습
데이터셋 경로: /Users/hjcha/Documents/NavieRAG실습/저작권판례데이터셋
ChromaDB 저장 경로: /Users/hjcha/Documents/NavieRAG실습/chroma_db


## Knowledge Base 구축: Step1. Data Load(Document Loader)

In [4]:
import pandas as pd
from langchain_core.documents import Document

case_raw_docs = []
qa_raw_docs = []

QA_EVAL_HOLDOUT_SIZE = 10
qa_full_df = pd.read_excel(DATASET_PATH / "Copyright_QA_GroundTrue.xlsx").dropna(subset=["Q", "A"]).reset_index(drop=True)
qa_kb_df = qa_full_df.iloc[:-QA_EVAL_HOLDOUT_SIZE].copy()
qa_eval_df = qa_full_df.iloc[-QA_EVAL_HOLDOUT_SIZE:].copy()
case_df = pd.read_excel(DATASET_PATH / "Copyright_case_law_Supreme.xlsx").dropna(subset=["판례내용"]).reset_index(drop=True)
kb_case_df = case_df.copy()

print(f"KB 판례 수: {len(kb_case_df)}")
print(f"Hybrid KB용 QA 수: {len(qa_kb_df)}")
print(f"평가용 QA holdout 수: {len(qa_eval_df)}")

KB 판례 수: 423
Hybrid KB용 QA 수: 21
평가용 QA holdout 수: 10


In [5]:
print("Evaluation QA holdout sample")
print("-----------------------------")
print(qa_eval_df[["Q", "A"]].head(3))

Evaluation QA holdout sample
-----------------------------
                                            Q  \
21      연예인의 이름을 상업적 행위에 이용할 경우, 손해배상책임을 지나요?   
22  다른 사람이 나의 저작물을 패러디하였는데 저작권 침해에 해당하는 것일까요?   
23  사진이나 영상에 연예인의 유행어를 이용하는 것도 침해에 해당하는 것일까요?   

                                                    A  
21  유명 연예인이나 운동선수 등은 자신의 성명이나 초상을 상업적으로 이용할 수 있는 권...  
22  저작권법상 허용되는 패러디에 대한 판단 기준이 명확하지는 않으나, 기존의 저작물에 ...  
23  짧은 유행어를 이용하는 경우는 저작권 침해로 보기는 어렵지만 퍼블리시티권 침해에 해...  


In [6]:
import logging
import re
from collections import OrderedDict

logger = logging.getLogger("rag_notebook")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

ARTICLE_PATTERN = re.compile(r"제\s*(\d+(?:의\d+)?)\s*조")
HO_PATTERN = re.compile(r"제\s*(\d+(?:의\d+)?)\s*항")

def normalize_legal_number(value: str) -> str:
    return str(value).replace(" ", "")

def extract_article_metadata(text: str) -> list[str]:
    return list(OrderedDict.fromkeys(normalize_legal_number(match) for match in ARTICLE_PATTERN.findall(text)))

def extract_ho_metadata(text: str) -> list[str]:
    return list(OrderedDict.fromkeys(normalize_legal_number(match) for match in HO_PATTERN.findall(text)))

def extract_query_article(question: str) -> str | None:
    match = ARTICLE_PATTERN.search(question)
    return normalize_legal_number(match.group(1)) if match else None

def clean_case_text(text: str) -> str:

    if text is None:
        return ""

    if "【이 유】" in text:
        text = text.split("【이 유】", 1)[1]

    text = re.split(r"【참조조문】", text)[0]
    text = re.split(r"【참조판례】", text)[0]
    text = re.sub(r"\n\s*\n", "\n", text)

    return text.strip()

In [7]:
# Case law dataset for knowledge base
for _, row in kb_case_df.iterrows():
    raw_text = str(row.get("판례내용", ""))
    cleaned_text = clean_case_text(raw_text)
    article_metadata = extract_article_metadata(cleaned_text)
    ho_metadata = extract_ho_metadata(cleaned_text)

    content = f"""
사건명: {row['사건명']}
법원: {row['법원명']}
선고일: {row['선고일자']}
사건번호: {row['사건번호']}

판례내용:
{cleaned_text}
"""

    case_raw_docs.append(
        Document(
            page_content=content,
            metadata={
                "document_type": "case_law",
                "case_number": str(row["사건번호"]).strip(),
                "court": str(row["법원명"]).strip(),
                "date": str(row["선고일자"]).strip(),
                "article": article_metadata,
                "ho": ho_metadata,
                "article_count": len(article_metadata),
            }
        )
    )

for idx, row in qa_kb_df.iterrows():
    combined_text = f"질문: {row['Q']}\n답변: {row['A']}"
    qa_raw_docs.append(
        Document(
            page_content=combined_text,
            metadata={
                "document_type": "qa",
                "case_number": f"QA-{idx}",
                "court": "QnA",
                "date": "N/A",
                "article": extract_article_metadata(combined_text),
                "ho": extract_ho_metadata(combined_text),
                "article_count": len(extract_article_metadata(combined_text)),
            }
        )
    )

print("Case KB documents:", len(case_raw_docs))
print("QA KB documents:", len(qa_raw_docs))
print("Case sample metadata:", case_raw_docs[-1].metadata)

Case KB documents: 423
QA KB documents: 21
Case sample metadata: {'document_type': 'case_law', 'case_number': '2003도7572', 'court': '대법원', 'date': '2004.07.22', 'article': ['4', '2'], 'ho': ['1'], 'article_count': 2}


In [8]:
print("Case law dataset")
print("-----------------------------")
print(f"[page_content]\n{case_raw_docs[0].page_content}")
print(f"\n[metadata]\n{case_raw_docs[0].metadata}")

Case law dataset
-----------------------------
[page_content]

사건명: 도메인이름 이전청구권 부존재확인, 손해배상(지)
법원: 대법원
선고일: 2022.04.14
사건번호: 2021다303134(본소),

판례내용:
【주 문】
 상고를 기각한다.
 상고비용은 원고(반소피고)가 부담한다.
 이유
 상고이유(상고이유서 제출기간이 지난 뒤에 제출된 참고서면의 기재는 이를 보충하는 범위에서)를 판단한다.
 1. 저작권 침해의 방조 관련 상고이유에 대한 판단
 가. 저작재산권자는 다른 사람에게 그 저작물의 이용을 허락할 수 있고, 그 이용을 허락 받은 자는 허락받은 이용 방법 및 조건의 범위 안에서 그 저작물을 이용할 수 있다. 또한 위 허락에 의하여 저작물을 이용할 수 있는 권리는 저작재산권자의 동의 없이 제3자에게 이를 양도할 수 없다(
 저작권법 제46조
 ). 따라서 저작권자로부터 저작물의 이용을 허락받지 못한 사람은 저작물을 복제 또는 사용할 수 없고, 저작물의 이용을 허락받은 사람도 허락받은 이용 방법 및 조건의 범위 안에서 그 저작물을 이용 · 양도할 수 있을 뿐이다.
 컴퓨터 프로그램 또는 그 라이선스의 판매 시 함께 제공되는 일련번호와 같은 제품키는 컴퓨터 프로그램을 설치 또는 사용할 권한이 있는가를 확인하는 수단인 기술적 보호조치로서, 누군가가 프로그램을 복제하고 그 복제행위가 컴퓨터 프로그램 저작권을 침해하는 행위에 해당한다면 이를 용이하게 하는 제품키의 복제 또는 배포행위는 위와 같은 행위를 용이하게 하는 행위로서 경우에 따라 프로그램 저작권 침해행위의 방조에 해당할 수 있다(
 대법원 2002. 6. 28. 선고 2001도2900 판결
 참조).
 나. 원심은 판시와 같은 이유를 들어 다음과 같이 판단하였다. ① 사용자가 별도의 라이선스 구입 없이 원고로부터 제품키만을 구입하여, 피고 사이트로부터 Windows 10Pro 또는 Home 프로그램을 다운로드 받으며 그 과정에서 제시되는 'Window

## Knowledge Base 구축: Step2. Chunking(Text Splitter)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "【", "판례내용:", "\n"],
)

case_docs = splitter.split_documents(case_raw_docs)
qa_docs = splitter.split_documents(qa_raw_docs)
docs_by_mode = {
    "case_only": case_docs,
    "case_plus_qa": case_docs + qa_docs,
}

docs = docs_by_mode["case_only"]

print(f"Case-only chunk count: {len(case_docs)}")
print(f"Case+QA chunk count: {len(docs_by_mode['case_plus_qa'])}")
print("\nChunk Samples\n")
for i, d in enumerate(case_docs[:5]):
    print(f"[Chunk {i}] ({len(d.page_content)} chars)")
    print(d.page_content[:100], "...\n")

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Case-only chunk count: 1543
Case+QA chunk count: 1578

Chunk Samples

[Chunk 0] (78 chars)
사건명: 도메인이름 이전청구권 부존재확인, 손해배상(지)
법원: 대법원
선고일: 2022.04.14
사건번호: 2021다303134(본소), ...

[Chunk 1] (5 chars)
판례내용: ...

[Chunk 2] (708 chars)
【주 문】
 상고를 기각한다.
 상고비용은 원고(반소피고)가 부담한다.
 이유
 상고이유(상고이유서 제출기간이 지난 뒤에 제출된 참고서면의 기재는 이를 보충하는 범위에서)를 판단한 ...

[Chunk 3] (931 chars)
대법원 2002. 6. 28. 선고 2001도2900 판결
 참조).
 나. 원심은 판시와 같은 이유를 들어 다음과 같이 판단하였다. ① 사용자가 별도의 라이선스 구입 없이 원고로 ...

[Chunk 4] (264 chars)
관련 법리와 기록에 의하면 이러한 원심판단은 정당하고, 거기에 필요한 심리를 다하지 아니하고 논리와 경험의 법칙을 위반하여 자유심증주의의 한계를 벗어나거나,
 인터넷주소자원에 관한 ...



## Knowledge Base 구축: Step3. Vectorizing(Embedding)

- 개념적으로는 두 단계이지만, Embedding + VectorDB 저장은 한번에 처리됨
- text-embedding-3-large은 3072차원  

In [10]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

sample_vector = embedding_model.embed_query("저작권 침해 판단 기준은?")
print(f"임베딩 차원수: {len(sample_vector)}")
print(f"벡터 앞 5개 값: {sample_vector[:5]}")

임베딩 차원수: 1536
벡터 앞 5개 값: [0.04510498046875, 0.050262451171875, 0.03668212890625, 0.0011882781982421875, 0.032440185546875]


## Knowledge Base 구축: Step4. Vector DB storage

In [11]:
# ChromaDB 저장을 위한 Path 설정
COLLECTION_NAMES = {
    "case_only": "copyright_rag_case_only",
    "case_plus_qa": "copyright_rag_case_plus_qa",
}
ACTIVE_MODE = "case_plus_qa"
print(f"ChromaDB 저장 경로: {CHROMA_PATH}")
print(f"활성 모드: {ACTIVE_MODE}")
print(f"컬렉션 이름: {COLLECTION_NAMES}")

ChromaDB 저장 경로: /Users/hjcha/Documents/NavieRAG실습/chroma_db
활성 모드: case_plus_qa
컬렉션 이름: {'case_only': 'copyright_rag_case_only', 'case_plus_qa': 'copyright_rag_case_plus_qa'}


In [12]:
from langchain_chroma import Chroma

def normalize_docs_for_chroma(documents):
    cleaned = []
    for doc in documents:
        metadata = doc.metadata.copy()
        if "article" in metadata and not metadata["article"]:
            metadata["article"] = None
        if "ho" in metadata and not metadata["ho"]:
            metadata["ho"] = None
        cleaned.append(Document(page_content=doc.page_content, metadata=metadata))
    return cleaned

def build_vector_store(documents, collection_name):
    vectorstore = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embedding_model,
        collection_name=collection_name,
    )
    normalized_docs = normalize_docs_for_chroma(documents)
    current_count = vectorstore._collection.count()
    if current_count != len(normalized_docs):
        if current_count > 0:
            vectorstore.delete_collection()
            vectorstore = Chroma(
                persist_directory=CHROMA_PATH,
                embedding_function=embedding_model,
                collection_name=collection_name,
            )
        vectorstore.add_documents(normalized_docs)
    return vectorstore

vectorstores_by_mode = {
    mode: build_vector_store(mode_docs, COLLECTION_NAMES[mode])
    for mode, mode_docs in docs_by_mode.items()
}
vectorstore = vectorstores_by_mode[ACTIVE_MODE]

for mode, store in vectorstores_by_mode.items():
    print(f"{mode}: {store._collection.count()} vectors")

case_only: 1543 vectors
case_plus_qa: 1578 vectors


In [13]:
# 보관된 vector DB 불러오기
# vectorstore = Chroma(
#     persist_directory=CHROMA_PATH,
#     embedding_function=embedding_model,
#     collection_name="copyright_rag",
# )

## Runtime: 질문 후보군

In [14]:
questions = [
    "저작권 침해가 성립하기 위한 요건은 무엇인가?",
    "저작물의 실질적 유사성 판단 기준은 무엇인가?",
    "저작권 보호 대상이 되는 저작물의 요건은 무엇인가?",
    "저작권 침해 판단에서 아이디어와 표현의 구별 기준은 무엇인가?",
    "저작권 침해 소송에서 법원이 고려하는 판단 기준은 무엇인가?"
]

for i, q in enumerate(questions, 1):
    print(f"Q{i}. {q}")

Q1. 저작권 침해가 성립하기 위한 요건은 무엇인가?
Q2. 저작물의 실질적 유사성 판단 기준은 무엇인가?
Q3. 저작권 보호 대상이 되는 저작물의 요건은 무엇인가?
Q4. 저작권 침해 판단에서 아이디어와 표현의 구별 기준은 무엇인가?
Q5. 저작권 침해 소송에서 법원이 고려하는 판단 기준은 무엇인가?


## Runtime: Step1. Query vectorization
- 동일한 embedding model을 사용해야 함

In [15]:
query = questions[2]

query_vector = embedding_model.embed_query(query)

print(f"Query: {query}")
print(f"\nQuery Vector Dimension: {len(query_vector)}")
print(f"Query Vector: {[round(v, 4) for v in query_vector[:5]]}")

Query: 저작권 보호 대상이 되는 저작물의 요건은 무엇인가?

Query Vector Dimension: 1536
Query Vector: [0.0673, 0.0629, -0.016, 0.017, 0.0495]


## Runtime: Step2. Retrieval
 ### BM25 + Vector Retriever

In [16]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [17]:
from langchain_community.retrievers import BM25Retriever

VECTOR_SEARCH_TOP_K = 20
BM25_SEARCH_TOP_K = 20
RERANK_TOP_K = 5
ENABLE_CONTEXT_DEDUP = False
DEDUP_SIMILARITY_THRESHOLD = 0.97
MAX_SEARCH_QUERIES = 2
SOFT_FILTER_BONUS = 0.15
QA_DOC_BONUS = 0.20
CASE_LAW_DOC_BONUS = 0.05
METADATA_PREFILTER_K = 8
RETRIEVAL_SAMPLE_SIZE = 5
FINAL_QA_DOCS = 2
FINAL_CASE_DOCS = 3

def build_bm25_retriever(corpus_documents):
    retriever = BM25Retriever.from_documents(corpus_documents)
    retriever.k = BM25_SEARCH_TOP_K
    return retriever

mode_resources = {
    mode: {
        "docs": mode_docs,
        "vectorstore": vectorstores_by_mode[mode],
        "bm25_retriever": build_bm25_retriever(mode_docs),
    }
    for mode, mode_docs in docs_by_mode.items()
}

logger.info(
    "Retrieval config | vector_top_k=%s | rerank_top_k=%s | dedup=%s | max_queries=%s",
    VECTOR_SEARCH_TOP_K,
    RERANK_TOP_K,
    ENABLE_CONTEXT_DEDUP,
    MAX_SEARCH_QUERIES,
)

print("Retrievers initialized for modes:", list(mode_resources.keys()))

INFO | Retrieval config | vector_top_k=20 | rerank_top_k=5 | dedup=False | max_queries=2


Retrievers initialized for modes: ['case_only', 'case_plus_qa']


In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

query_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template="""당신은 법률 검색 질의 확장기입니다.
사용자 질문에 대해 검색 품질이 높은 한국어 재작성 질의 1개만 생성하세요.

조건:
- 원문 질문과 의미가 동일해야 합니다.
- 조문 번호가 있으면 반드시 유지하세요.
- 불필요한 설명 없이 질의 한 줄만 출력하세요.

질문: {question}""",
)

llm_chain = multi_query_prompt | query_llm

def expand_queries(question: str) -> list[str]:
    generated = llm_chain.invoke({"question": question})

    if hasattr(generated, "content"):
        text = generated.content
    elif isinstance(generated, dict):
        text = generated.get("text") or generated.get("queries") or next(iter(generated.values()))
    else:
        text = str(generated)

    expanded_queries = [line.strip("- ").strip() for line in str(text).splitlines() if line.strip()]
    queries = list(OrderedDict.fromkeys([question, *expanded_queries]))
    return queries[:MAX_SEARCH_QUERIES]

queries = expand_queries(query)
print("\nExpanded Queries:")
for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")


Expanded Queries:
1. 저작권 보호 대상이 되는 저작물의 요건은 무엇인가?
2. 저작권 보호 대상 저작물의 요건은 무엇인가?


In [19]:
def log_documents(title: str, documents, limit: int = 5):
    logger.info("%s | count=%s", title, len(documents))
    for idx, doc in enumerate(documents[:limit], 1):
        preview = doc.page_content[:160].replace("\n", " ")
        logger.info(
            "%s #%s | case=%s | type=%s | article=%s | preview=%s",
            title,
            idx,
            doc.metadata.get("case_number"),
            doc.metadata.get("document_type"),
            doc.metadata.get("article"),
            preview,
        )

def has_article_match(doc, article_number: str | None) -> bool:
    if not article_number:
        return False
    return article_number in (doc.metadata.get("article") or [])

def document_type_bonus(doc, mode_name: str) -> float:
    doc_type = doc.metadata.get("document_type")
    if mode_name == "case_plus_qa" and doc_type == "qa":
        return QA_DOC_BONUS
    if doc_type == "case_law":
        return CASE_LAW_DOC_BONUS
    return 0.0

def metadata_soft_score(doc, article_number: str | None):
    if not article_number:
        return 0.0
    return SOFT_FILTER_BONUS if has_article_match(doc, article_number) else 0.0

def collect_vector_candidates(search_query: str, article_number: str | None, mode_name: str):
    vectorstore = mode_resources[mode_name]["vectorstore"]
    base_docs = vectorstore.similarity_search(search_query, k=VECTOR_SEARCH_TOP_K)
    if not article_number:
        return base_docs
    expanded_docs = vectorstore.similarity_search(search_query, k=VECTOR_SEARCH_TOP_K * 2)
    prioritized_docs = [doc for doc in expanded_docs if has_article_match(doc, article_number)]
    merged_map = OrderedDict()
    for doc in prioritized_docs[:METADATA_PREFILTER_K] + base_docs:
        key = (doc.metadata.get("case_number"), doc.page_content)
        merged_map[key] = doc
    return list(merged_map.values())

def retrieve_candidates(question: str, mode_name: str = ACTIVE_MODE, verbose: bool = True):
    article_number = extract_query_article(question)
    search_queries = expand_queries(question)
    bm25_retriever = mode_resources[mode_name]["bm25_retriever"]
    scored_candidates = {}

    if article_number:
        logger.info("Soft metadata bias enabled | mode=%s | article=%s", mode_name, article_number)

    for q in search_queries:
        vector_docs = collect_vector_candidates(q, article_number, mode_name)
        bm25_docs = bm25_retriever.invoke(q)
        results = vector_docs + bm25_docs
        if verbose:
            print(f"\n[{mode_name}] Query: {q}")
            print(f"검색 결과: {len(results)}개")
        for rank, doc in enumerate(results):
            key = (doc.metadata.get("case_number"), doc.page_content)
            base_score = 1 / (rank + 1)
            bonus = metadata_soft_score(doc, article_number) + document_type_bonus(doc, mode_name)
            previous = scored_candidates.get(key, {"score": 0.0})["score"]
            scored_candidates[key] = {
                "doc": doc,
                "score": max(previous, base_score + bonus),
            }

    ranked_candidates = sorted(scored_candidates.values(), key=lambda item: item["score"], reverse=True)
    unique_docs = [item["doc"] for item in ranked_candidates[:VECTOR_SEARCH_TOP_K]]

    if verbose:
        print(f"\n[{mode_name}] 후보 문서 수:", len(unique_docs))
        log_documents(f"retrieved documents [{mode_name}]", unique_docs, limit=RETRIEVAL_SAMPLE_SIZE)
        print(f"\n[{mode_name}] Retrieval 샘플 확인")
        for idx, doc in enumerate(unique_docs[:RETRIEVAL_SAMPLE_SIZE], 1):
            print(f"[{idx}] case_number={doc.metadata.get('case_number')} | type={doc.metadata.get('document_type')} | article={doc.metadata.get('article')}")
            print(doc.page_content[:300])
            print("-" * 80)

    return {"mode": mode_name, "article_filter": article_number, "queries": search_queries, "documents": unique_docs}

retrieval_result = retrieve_candidates(query, mode_name=ACTIVE_MODE)
queries = retrieval_result["queries"]
unique_docs = retrieval_result["documents"]

INFO | retrieved documents [case_plus_qa] | count=20
INFO | retrieved documents [case_plus_qa] #1 | case=93다3073,93다3080 | type=case_law | article=None | preview=본소에 관한 논지는 모두 이유 없다. 2. 반소에 관하여 저작권법에 의하여 보호되는 저작물은 학문과 예술에 관하여 사람의 정신적 노력에 의하여 얻어진 사상 또는 감정의 창작적 표현물이어야 하는 것이고 ( 당원 1979.12.28. 선고 79도1482 판결 ; 1990.10.23. 선고 
INFO | retrieved documents [case_plus_qa] #2 | case=2009도6073 | type=case_law | article=['4'] | preview=【주 문】  원심판결을 파기하고, 사건을 서울중앙지방법원에 환송한다.  이 유  상고이유를 판단한다.  구 저작권법(2006. 12. 28. 법률 제8101호로 전문 개정되기 전의 것, 이하 같다) 제4조 제1항 제8호  는 "지도, 도표, 설계도, 약도, 모형 그 밖의 도형저작물"을 
INFO | retrieved documents [case_plus_qa] #3 | case=2007다63409 | type=case_law | article=['5'] | preview=소정의 2차적저작물로 보호받기 위하여는 원저작물을 기초로 하되 원저작물과 실질적 유사성을 유지하고 이것에 사회통념상 새로운 저작물이 될 수 있을 정도의 수정·증감을 가하여 새로운 창작성을 부가하여야 하는 것이므로 ( 대법원 2002. 1. 25. 선고 99도863 판결 , 대법원 200
INFO | retrieved documents [case_plus_qa] #4 | case=2011도3599 | type=case_law | article=['5'] | preview=【주 문】상고를 모두 기각한다. 이    유 상고이유를 판단한다. 1. 저작


[case_plus_qa] Query: 저작권 보호 대상이 되는 저작물의 요건은 무엇인가?
검색 결과: 40개

[case_plus_qa] Query: 저작권 보호 대상 저작물의 요건은 무엇인가?
검색 결과: 40개

[case_plus_qa] 후보 문서 수: 20

[case_plus_qa] Retrieval 샘플 확인
[1] case_number=93다3073,93다3080 | type=case_law | article=None
본소에 관한 논지는 모두 이유 없다.
2. 반소에 관하여
저작권법에 의하여 보호되는 저작물은 학문과 예술에 관하여 사람의 정신적 노력에 의하여 얻어진 사상 또는 감정의 창작적 표현물이어야 하는 것이고
(
당원 1979.12.28. 선고 79도1482 판결
;
1990.10.23. 선고 90다카8845 판결
),
따라서 저작권법이 보호하고 있는 것은 사상, 감정을 말, 문자, 음, 색 등에 의하여 구체적으로 외부에 표현된 창작적인 표현형식이고, 그 표현되어 있는 내용 즉 아이디어나 이론 등의 사상 및 감정 그 자체는 설사 그것이 독창
--------------------------------------------------------------------------------
[2] case_number=2009도6073 | type=case_law | article=['4']
【주 문】
 원심판결을 파기하고, 사건을 서울중앙지방법원에 환송한다.
 이 유
 상고이유를 판단한다.
 구 저작권법(2006. 12. 28. 법률 제8101호로 전문 개정되기 전의 것, 이하 같다) 제4조 제1항 제8호
 는 "지도, 도표, 설계도, 약도, 모형 그 밖의 도형저작물"을 저작물의 하나로 예시하고 있는데, 이와 같은 도형저작물은 예술성의 표현보다는 기능이나 실용적인 사상의 표현을 주된 목적으로 하는 이른바 기능적 저작물로서, 기능적 저작물은 그 표현하고자 하는 기능 또는 실용적인 사상이 속하는 분야에서의 일반적인 표현방
--------------

### Reranking

In [20]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-large")

def cosine_similarity(vec_a, vec_b):
    return float(np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)))

def deduplicate_context(documents, similarity_threshold=DEDUP_SIMILARITY_THRESHOLD):
    if not documents or not ENABLE_CONTEXT_DEDUP:
        return documents
    embeddings = np.array(embedding_model.embed_documents([doc.page_content for doc in documents]))
    unique_docs = []
    unique_vectors = []
    for doc, vector in zip(documents, embeddings):
        if not unique_vectors:
            unique_docs.append(doc)
            unique_vectors.append(vector)
            continue
        similarities = [cosine_similarity(vector, existing_vector) for existing_vector in unique_vectors]
        if max(similarities) < similarity_threshold:
            unique_docs.append(doc)
            unique_vectors.append(vector)
    return unique_docs

def compose_final_documents(reranked_docs, mode_name: str, top_k: int = RERANK_TOP_K):
    if mode_name != "case_plus_qa":
        return reranked_docs[:top_k]

    qa_docs = [doc for doc in reranked_docs if doc.metadata.get("document_type") == "qa"]
    case_docs = [doc for doc in reranked_docs if doc.metadata.get("document_type") == "case_law"]
    selected = []
    selected.extend(case_docs[:FINAL_CASE_DOCS])
    selected.extend(qa_docs[:FINAL_QA_DOCS])
    if len(selected) < top_k:
        seen = {(d.metadata.get("case_number"), d.page_content) for d in selected}
        for doc in reranked_docs:
            key = (doc.metadata.get("case_number"), doc.page_content)
            if key not in seen:
                selected.append(doc)
                seen.add(key)
            if len(selected) == top_k:
                break
    return selected[:top_k]

def rerank_documents(question: str, candidate_docs, mode_name: str = ACTIVE_MODE, top_k: int = RERANK_TOP_K):
    if not candidate_docs:
        return []
    pairs = [(question, doc.page_content) for doc in candidate_docs]
    scores = reranker.predict(pairs)
    scored_docs = []
    for score, doc in zip(scores, candidate_docs):
        adjusted_score = float(score) + document_type_bonus(doc, mode_name)
        scored_docs.append((adjusted_score, doc))
    scored_docs.sort(key=lambda item: item[0], reverse=True)
    reranked_docs = [doc for _, doc in scored_docs]
    post_processed_docs = deduplicate_context(reranked_docs)
    selected_docs = compose_final_documents(post_processed_docs, mode_name=mode_name, top_k=top_k)
    log_documents(f"reranked results [{mode_name}]", selected_docs, limit=RETRIEVAL_SAMPLE_SIZE)
    return selected_docs

selected_docs = rerank_documents(query, unique_docs, mode_name=ACTIVE_MODE)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 11215.78it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO | Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]
INFO | reranked results [case_plus_qa] | count=5
INFO | reranked results [case_plus_qa] #1 | case=2011도3599 | type=case_law | article=['5'] | preview=【주 문】상고를 모두 기각한다. 이    유 상고이유를 판단한다. 1. 저작권법 제5조 제1항 은 ‘원저작물을 번역·편곡·변형·각색·영상제작 그 밖의 방법으로 작성한 창작물’을 ‘2차적저작물’이라고 규정하고 있는바, 2차적저작물이 되기 위해서는 원저작물을 기초로 수정·증감이 가해지되 원
INFO | reranked results [case_plus_qa] #2 | case=2009도6073 | type=case_law | article=['4'] | preview=【주 문】  원심판결을 파기하고, 사건을 서울중앙지방법원에 환송한다.  이 유  상고이유를 판단한다.  구 저작권법(2006. 12. 28. 법

In [21]:
# 상위 context 선택
top_k = RERANK_TOP_K
print(f"\n[{ACTIVE_MODE}] Top {top_k} Context 선택", len(selected_docs))
for i, doc in enumerate(selected_docs, 1):
    print(f"\n[Document {i}]")
    print("metadata:", doc.metadata)
    print("content preview:", doc.page_content[:300])
    print("-" * 80)


[case_plus_qa] Top 5 Context 선택 5

[Document 1]
metadata: {'court': '대법원', 'date': '2013.08.22', 'article': ['5'], 'article_count': 1, 'case_number': '2011도3599', 'document_type': 'case_law', 'ho': ['1']}
content preview: 【주 문】상고를 모두 기각한다.
이    유
상고이유를 판단한다.
1.
저작권법 제5조 제1항
은 ‘원저작물을 번역·편곡·변형·각색·영상제작 그 밖의 방법으로 작성한 창작물’을 ‘2차적저작물’이라고 규정하고 있는바, 2차적저작물이 되기 위해서는 원저작물을 기초로 수정·증감이 가해지되 원저작물과 실질적 유사성을 유지하여야 한다. 따라서 어문저작물인 원저작물을 기초로 하여 이를 요약한 요약물이 원저작물과 실질적인 유사성이 없는 별개의 독립적인 새로운 저작물이 된 경우에는 원저작물 저작권자의 2차적저작물작성권을 침해한 것으로 되지는 아니
--------------------------------------------------------------------------------

[Document 2]
metadata: {'court': '대법원', 'ho': ['1'], 'case_number': '2009도6073', 'article': ['4'], 'document_type': 'case_law', 'date': '2011.05.13', 'article_count': 1}
content preview: 【주 문】
 원심판결을 파기하고, 사건을 서울중앙지방법원에 환송한다.
 이 유
 상고이유를 판단한다.
 구 저작권법(2006. 12. 28. 법률 제8101호로 전문 개정되기 전의 것, 이하 같다) 제4조 제1항 제8호
 는 "지도, 도표, 설계도, 약도, 모형 그 밖의 도형저작물"을 저작물의 하나로 예시하고 있는데, 이와 같은 도형저작물은 예술성의 표현보다는 기능이나

## Runtime: Step3. LLM request (RAG Chain 구성)

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

answer_llm = ChatOpenAI(model="gpt-5.2", temperature=0.05)

prompt = ChatPromptTemplate.from_template("""
당신은 저작권법 전문가입니다.

규칙:
1. 반드시 제공된 참고 문맥의 내용만 사용하세요.
2. 답변에는 관련 조문 번호(예: 제2조, 제35조의3)와 Source를 반드시 포함하세요.
3. 참고 문맥에 없는 정보는 추측하지 마세요.
4. 답을 찾을 수 없으면 반드시 정확히 "정보를 찾을 수 없습니다"라고 답하세요.
5. 질문이 QA형이면 불필요하게 장황하게 쓰지 말고, 문맥에 근거한 핵심 답을 먼저 간결하게 제시하세요.
6. QA 문서와 판례 문서가 함께 있으면, QA 문서의 직접 답변을 우선 활용하되 판례 Source도 함께 제시하세요.

[참고 문맥]
{context}

[질문]
{question}

[답변 형식]
- 핵심 답변
- 근거 조문/판례
- Source
""")

def format_docs(documents):
    formatted = []
    for idx, doc in enumerate(documents, 1):
        source = doc.metadata.get("case_number", "N/A")
        articles = doc.metadata.get("article") or []
        ho_list = doc.metadata.get("ho") or []
        article_text = ", ".join(f"제{article}조" for article in articles[:5]) or "조문 정보 없음"
        ho_text = ", ".join(f"제{ho}항" for ho in ho_list[:5]) or "항 정보 없음"
        formatted.append(f"[Source {idx}] case_number={source} | document_type={doc.metadata.get('document_type')} | article={article_text} | ho={ho_text}\n{doc.page_content}")
    return "\n\n".join(formatted)

rag_chain = prompt | answer_llm | StrOutputParser()

def generate_answer(question: str, selected_docs, mode_name: str = ACTIVE_MODE):
    context = format_docs(selected_docs)
    logger.info("final context | mode=%s | chars=%s", mode_name, len(context))
    logger.info("final context preview | %s", context[:1500].replace("\n", " "))
    answer = rag_chain.invoke({"context": context, "question": question})
    return answer, context

print("RAG Chain 구성 완료")
answer, context = generate_answer(query, selected_docs, mode_name=ACTIVE_MODE)
print("\n최종 Context 길이:", len(context))

INFO | final context | mode=case_plus_qa | chars=9918
INFO | final context preview | [Source 1] case_number=2011도3599 | document_type=case_law | article=제5조 | ho=제1항 【주 문】상고를 모두 기각한다. 이    유 상고이유를 판단한다. 1. 저작권법 제5조 제1항 은 ‘원저작물을 번역·편곡·변형·각색·영상제작 그 밖의 방법으로 작성한 창작물’을 ‘2차적저작물’이라고 규정하고 있는바, 2차적저작물이 되기 위해서는 원저작물을 기초로 수정·증감이 가해지되 원저작물과 실질적 유사성을 유지하여야 한다. 따라서 어문저작물인 원저작물을 기초로 하여 이를 요약한 요약물이 원저작물과 실질적인 유사성이 없는 별개의 독립적인 새로운 저작물이 된 경우에는 원저작물 저작권자의 2차적저작물작성권을 침해한 것으로 되지는 아니하는데 ( 대법원 2010. 2. 11. 선고 2007다63409 판결 등 참조), 여기서 요약물이 그 원저작물과 사이에 실질적인 유사성이 있는지 여부는, 요약물이 원저작물의 기본으로 되는 개요, 구조, 주된 구성 등을 그대로 유지하고 있는지 여부, 요약물이 원저작물을 이루는 문장들 중 일부만을 선택하여 발췌한 것이거나 발췌한 문장들의 표현을 단순히 단축한 정도에 불과한지 여부, 원저작물과 비교한 요약물의 상대적인 분량, 요약물의 원저작물에 대한 대체가능성 여부 등을 종합적으로 고려하여 판단해야 한다. 한편 저작권의 보호 대상은 인간의 사상 또는 감정을 말, 문자, 음, 색 등에 의하여 구체적으로 외부에 표현한 창작적인 표현형식이고, 거기에 표현되어 있는 내용 즉 아이디어나 이론 등의 사상 또는 감정 그 자체는 원칙적으로 저작권의 보호 대상이 아니므로, 저작권의 침해 여부를 가리기 위하여 두 저작물 사이에 실질적인 유사성이 있는지 여부를 판단함에 있어서도 창작적인 표현형식에 해당하는 것만을 가지고 대비해 보아야 하고, 표현형식이 아닌 사상 또

RAG Chain 구성 완료

최종 Context 길이: 9918


In [23]:
# 최종 프롬프트 확인
final_prompt = prompt.invoke({"context": format_docs(selected_docs), "question": query})
print(final_prompt.messages[0].content)
print(answer)


당신은 저작권법 전문가입니다.

규칙:
1. 반드시 제공된 참고 문맥의 내용만 사용하세요.
2. 답변에는 관련 조문 번호(예: 제2조, 제35조의3)와 Source를 반드시 포함하세요.
3. 참고 문맥에 없는 정보는 추측하지 마세요.
4. 답을 찾을 수 없으면 반드시 정확히 "정보를 찾을 수 없습니다"라고 답하세요.
5. 질문이 QA형이면 불필요하게 장황하게 쓰지 말고, 문맥에 근거한 핵심 답을 먼저 간결하게 제시하세요.
6. QA 문서와 판례 문서가 함께 있으면, QA 문서의 직접 답변을 우선 활용하되 판례 Source도 함께 제시하세요.

[참고 문맥]
[Source 1] case_number=2011도3599 | document_type=case_law | article=제5조 | ho=제1항
【주 문】상고를 모두 기각한다.
이    유
상고이유를 판단한다.
1.
저작권법 제5조 제1항
은 ‘원저작물을 번역·편곡·변형·각색·영상제작 그 밖의 방법으로 작성한 창작물’을 ‘2차적저작물’이라고 규정하고 있는바, 2차적저작물이 되기 위해서는 원저작물을 기초로 수정·증감이 가해지되 원저작물과 실질적 유사성을 유지하여야 한다. 따라서 어문저작물인 원저작물을 기초로 하여 이를 요약한 요약물이 원저작물과 실질적인 유사성이 없는 별개의 독립적인 새로운 저작물이 된 경우에는 원저작물 저작권자의 2차적저작물작성권을 침해한 것으로 되지는 아니하는데
(
대법원 2010. 2. 11. 선고 2007다63409 판결
등 참조),
여기서 요약물이 그 원저작물과 사이에 실질적인 유사성이 있는지 여부는, 요약물이 원저작물의 기본으로 되는 개요, 구조, 주된 구성 등을 그대로 유지하고 있는지 여부, 요약물이 원저작물을 이루는 문장들 중 일부만을 선택하여 발췌한 것이거나 발췌한 문장들의 표현을 단순히 단축한 정도에 불과한지 여부, 원저작물과 비교한 요약물의 상대적인 분량, 요약물의 원저작물에 대한 대체가능성 여부 등을 종합적으로 고려하여 판단해야 한다.
한편 저작권의 보호 대상은 인간의 

## Runtime: Step4. 답변 출력

In [24]:
import langchain
langchain.debug = True

answer, context = generate_answer(query, selected_docs, mode_name=ACTIVE_MODE)

print(f"질문: {query}")
print(f"\n답변:\n{answer}")

INFO | final context | mode=case_plus_qa | chars=9918
INFO | final context preview | [Source 1] case_number=2011도3599 | document_type=case_law | article=제5조 | ho=제1항 【주 문】상고를 모두 기각한다. 이    유 상고이유를 판단한다. 1. 저작권법 제5조 제1항 은 ‘원저작물을 번역·편곡·변형·각색·영상제작 그 밖의 방법으로 작성한 창작물’을 ‘2차적저작물’이라고 규정하고 있는바, 2차적저작물이 되기 위해서는 원저작물을 기초로 수정·증감이 가해지되 원저작물과 실질적 유사성을 유지하여야 한다. 따라서 어문저작물인 원저작물을 기초로 하여 이를 요약한 요약물이 원저작물과 실질적인 유사성이 없는 별개의 독립적인 새로운 저작물이 된 경우에는 원저작물 저작권자의 2차적저작물작성권을 침해한 것으로 되지는 아니하는데 ( 대법원 2010. 2. 11. 선고 2007다63409 판결 등 참조), 여기서 요약물이 그 원저작물과 사이에 실질적인 유사성이 있는지 여부는, 요약물이 원저작물의 기본으로 되는 개요, 구조, 주된 구성 등을 그대로 유지하고 있는지 여부, 요약물이 원저작물을 이루는 문장들 중 일부만을 선택하여 발췌한 것이거나 발췌한 문장들의 표현을 단순히 단축한 정도에 불과한지 여부, 원저작물과 비교한 요약물의 상대적인 분량, 요약물의 원저작물에 대한 대체가능성 여부 등을 종합적으로 고려하여 판단해야 한다. 한편 저작권의 보호 대상은 인간의 사상 또는 감정을 말, 문자, 음, 색 등에 의하여 구체적으로 외부에 표현한 창작적인 표현형식이고, 거기에 표현되어 있는 내용 즉 아이디어나 이론 등의 사상 또는 감정 그 자체는 원칙적으로 저작권의 보호 대상이 아니므로, 저작권의 침해 여부를 가리기 위하여 두 저작물 사이에 실질적인 유사성이 있는지 여부를 판단함에 있어서도 창작적인 표현형식에 해당하는 것만을 가지고 대비해 보아야 하고, 표현형식이 아닌 사상 또

질문: 저작권 보호 대상이 되는 저작물의 요건은 무엇인가?

답변:
- 핵심 답변  
저작권법상 보호 대상이 되는 저작물이 되려면 **① ‘인간의 사상 또는 감정’을 ② ‘표현한’ ③ ‘창작물’**이어야 합니다. 또한 여기서 말하는 창작성이란 **완전한 독창성까지 요구되는 것은 아니고**, **단순 모방이 아닌 저작자 자신의 독자적인 사상 또는 감정의 표현**이 담겨 **기존 작품과 구별될 정도**이면足합니다. (아이디어 자체가 아니라 **창작적인 표현형식**이 보호 대상입니다.)

- 근거 조문/판례  
  - **저작권법 제2조 제1호**: “저작물”은 **인간의 사상 또는 감정을 표현한 창작물**  
  - **대법원 1994. 12. 2. 선고 94도2238 판결**: 창작성은 완전한 독창성이 아니라 **단순 모방이 아니고 저작자 자신의 독자적인 사상 또는 감정의 표현**을 담아 **구별 가능**하면 충분  
  - **대법원 2011. 4. 28. 선고 2011도3599 판결**: 저작권 보호는 **창작적인 표현형식**에 미치며 **아이디어·이론 등 내용 자체는 원칙적으로 보호 대상이 아님**

- Source  
  - Source 4 (저작권법 제2조)  
  - Source 3 (94도2238)  
  - Source 1 (2011도3599)


## RAG 성능 평가

In [25]:
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

eval_sample_size = min(10, len(qa_eval_df))
eval_sample_df = qa_eval_df[["Q", "A"]].head(eval_sample_size).copy()
eval_sample_df.columns = ["question", "reference"]

print(f"RAGAS 평가 질문 수: {len(eval_sample_df)}")
eval_sample_df.head()

RAGAS 평가 질문 수: 10


/var/folders/5b/_vr668qd1qjfth5qbkggmk0m0000gn/T/ipykernel_52576/3220518135.py:3: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
/var/folders/5b/_vr668qd1qjfth5qbkggmk0m0000gn/T/ipykernel_52576/3220518135.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
/var/folders/5b/_vr668qd1qjfth5qbkggmk0m0000gn/T/ipykernel_52576/3220518135.py:3: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instea

,question,reference
21,"연예인의 이름을 상업적 행위에 이용할 경우, 손해배상책임을 지나요?",유명 연예인이나 운동선수 등은 자신의 성명이나 초상을 상업적으로 이용할 수 있는 권...
22,다른 사람이 나의 저작물을 패러디하였는데 저작권 침해에 해당하는 것일까요?,"저작권법상 허용되는 패러디에 대한 판단 기준이 명확하지는 않으나, 기존의 저작물에 ..."
23,사진이나 영상에 연예인의 유행어를 이용하는 것도 침해에 해당하는 것일까요?,짧은 유행어를 이용하는 경우는 저작권 침해로 보기는 어렵지만 퍼블리시티권 침해에 해...
24,"이미 사망한 유명인을 모델로 하여 영화나 드라마를 제작하려고 하는데, 유족의 허락을...",사망한 유명인의 초상이나 성명을 상업적으로 이용하는 것은 퍼블리시티권과 관련하여 법...
25,저작자가 명시적 또는 묵시적으로 동의한 범위 내에서 저작물을 변경할 경우 동일성유지...,저작자가 명시적으로 동의한 범위 내에서 저작물을 변경하는 것은 아무런 문제가 되지 ...


In [26]:
def answer_with_rag(question, mode_name: str = ACTIVE_MODE):
    retrieval_result = retrieve_candidates(question, mode_name=mode_name, verbose=False)
    selected_docs = rerank_documents(question, retrieval_result["documents"], mode_name=mode_name)
    response, _ = generate_answer(question, selected_docs, mode_name=mode_name)
    return {
        "mode": mode_name,
        "user_input": question,
        "retrieved_contexts": [doc.page_content for doc in selected_docs],
        "response": response,
    }

In [27]:
def evaluate_ragas(eval_frame, mode_name: str, sample_size=eval_sample_size):
    eval_samples = []
    for row in eval_frame.head(sample_size).itertuples(index=False):
        result = answer_with_rag(row.question, mode_name=mode_name)
        result["reference"] = row.reference
        eval_samples.append(result)
    evaluation_dataset = EvaluationDataset.from_list(eval_samples)
    print(f"[{mode_name}] 평가 데이터셋 생성 완료: {len(eval_samples)}건")
    return evaluation_dataset

evaluation_datasets = {
    mode: evaluate_ragas(eval_sample_df, mode_name=mode)
    for mode in ["case_only", "case_plus_qa"]
}

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]
INFO | reranked results [case_only] | count=5
INFO | reranked results [case_only] #1 | case=2012도13718 | type=case_law | article=['20', '21', '307', '37'] | preview=사건명: 명예훼손(일부예비적죄명:모욕)·저작권법위반 법원: 대법원 선고일: 2014.09.04 사건번호: 2012도13718
INFO | reranked results [case_only] #2 | case=96다2460 | type=case_law | article=['5', '64'] | preview=사건명: 손해배상(자) 법원: 대법원 선고일: 1997.05.28 사건번호: 96다2460
INFO | reranked results [case_only] #3 | case=98다41216 | type=case_law | article=['91', '93', '95', '97'] | preview=사건명: 손해배상(지) 법원: 대법원 선고일: 1999.05.25 사건번호: 98다41216
INFO | reranked results [case_only] #4 | case=2002다18244 | type=case_law | article=['93'] | preview=사건명: 손해배상(지) 법원: 대법원 선고일: 2004.06.11 사건번호: 2002다18244
INFO | reranked results [case_only] #5 | case=2004다18736 | type=case_law | article=['5'] | preview=사건명: 손해배상(지) 법원: 대법원 선고일: 2004.07.08 사건번호: 2004다18736
INFO | final context | mode=case_only | chars=744
INFO | final context preview | [

[case_only] 평가 데이터셋 생성 완료: 10건


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]
INFO | reranked results [case_plus_qa] | count=5
INFO | reranked results [case_plus_qa] #1 | case=98다41216 | type=case_law | article=['91', '93', '95', '97'] | preview=사건명: 손해배상(지) 법원: 대법원 선고일: 1999.05.25 사건번호: 98다41216
INFO | reranked results [case_plus_qa] #2 | case=2016다276467 | type=case_law | article=['4', '2'] | preview=사건명: 손해배상(지)[부정경쟁방지 및 영업비밀보호에 관한 법률 제2조 제1호 (카)목의 성과물 도용 부정경쟁행위의 해당 여부] 법원: 대법원 선고일: 2020.03.26 사건번호: 2016다276467
INFO | reranked results [case_plus_qa] #3 | case=2011도1435 | type=case_law | article=['102', '103', '254', '140', '136', '141'] | preview=같은 취지의 원심판단은 정당한 것으로 수긍할 수 있고, 거기에 상고이유의 주장과 같은 공소사실의 특정에 관한 법리를 오해하는 등의 위법이 없다. 3. 피고인 3, 피고인 4 주식회사, 피고인 7, 피고인 8 주식회사의 상습성에 관한 상고이유에 대하여 가. 구 저작권법 제140조 본문에서
INFO | reranked results [case_plus_qa] #4 | case=QA-20 | type=qa | article=None | preview= 답변: 연예인 사진을 허락 없이 이용하는 행위는 퍼블리시티권 침해에 해당할 수 있으니 주의해야 합니다.연예인 등 유명인의 경우에는 인격적 이익보다는 경제적 이익이 주로 문제된다고 하는 특수성이 있다는 점

[case_plus_qa] 평가 데이터셋 생성 완료: 10건


In [28]:
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))

ragas_results = {}
ragas_frames = {}
for mode, dataset in evaluation_datasets.items():
    ragas_results[mode] = evaluate(
        dataset=dataset,
        metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
        llm=evaluator_llm,
    )
    ragas_frames[mode] = ragas_results[mode].to_pandas()

ragas_frames[ACTIVE_MODE].head()

/var/folders/5b/_vr668qd1qjfth5qbkggmk0m0000gn/T/ipykernel_52576/157828713.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
Evaluating: 100%|██████████| 30/30 [01:09<00:00,  2.32s/it]


,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1)
0,"연예인의 이름을 상업적 행위에 이용할 경우, 손해배상책임을 지나요?",[사건명: 손해배상(지)\n법원: 대법원\n선고일: 1999.05.25\n사건번호:...,- **핵심 답변** \n 연예인의 **이름(성명)이나 초상**을 **상업적(영...,유명 연예인이나 운동선수 등은 자신의 성명이나 초상을 상업적으로 이용할 수 있는 권...,1.000,1.000000,0.18
1,다른 사람이 나의 저작물을 패러디하였는데 저작권 침해에 해당하는 것일까요?,[【주 문】상고를 기각한다. 상고비용은 피고(반소원고)가 부담한다.\n이 유...,- 핵심 답변 \n참고 문맥만으로는 “패러디”가 저작권(복제권 또는 2차적저작물작...,"저작권법상 허용되는 패러디에 대한 판단 기준이 명확하지는 않으나, 기존의 저작물에 ...",1.000,0.818182,0.38
2,사진이나 영상에 연예인의 유행어를 이용하는 것도 침해에 해당하는 것일까요?,"[\n상고이유를 판단한다. 1. 원심은, 피고인들이 2002년 한·일월드컵 당시 널...",- 핵심 답변 \n참고 문맥에는 “연예인의 유행어”를 사진·영상에 이용하는 경우가...,짧은 유행어를 이용하는 경우는 저작권 침해로 보기는 어렵지만 퍼블리시티권 침해에 해...,1.000,0.500000,0.14
3,"이미 사망한 유명인을 모델로 하여 영화나 드라마를 제작하려고 하는데, 유족의 허락을...","[【주 문】원심판결을 파기하고, 사건을 서울중앙지방법원 합의부에 환송한다.\n이 ...",- 핵심 답변 \n판례의 입장이 통일되지 않아 **유족(상속인)의 허락을 얻는 것...,사망한 유명인의 초상이나 성명을 상업적으로 이용하는 것은 퍼블리시티권과 관련하여 법...,0.875,1.000000,0.57
4,저작자가 명시적 또는 묵시적으로 동의한 범위 내에서 저작물을 변경할 경우 동일성유지...,[【주 문】상고를 모두 기각한다. 상고비용은 원고들이 부담한다.\n이 유\n...,- 핵심 답변 \n아니요. 저작자가 **명시적 또는 묵시적으로 동의한 범위 내에서...,저작자가 명시적으로 동의한 범위 내에서 저작물을 변경하는 것은 아무런 문제가 되지 ...,0.800,1.000000,0.27


In [29]:
metric_columns = ["context_recall", "faithfulness", "factual_correctness(mode=f1)"]

comparison_rows = []
for mode, ragas_df in ragas_frames.items():
    available_metric_columns = [col for col in metric_columns if col in ragas_df.columns]
    ragas_summary = ragas_df[available_metric_columns].mean()
    row = {"mode": mode}
    row.update({metric: ragas_summary.get(metric) for metric in available_metric_columns})
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,mode,context_recall,faithfulness,factual_correctness(mode=f1)
0,case_only,0.832218,0.458333,0.198
1,case_plus_qa,0.927500,0.509596,0.189


In [30]:
mode_to_inspect = ACTIVE_MODE
ragas_frames[mode_to_inspect][["user_input", *[col for col in metric_columns if col in ragas_frames[mode_to_inspect].columns]]].head(eval_sample_size)

,user_input,context_recall,faithfulness,factual_correctness(mode=f1)
0,"연예인의 이름을 상업적 행위에 이용할 경우, 손해배상책임을 지나요?",1.000,1.000000,0.18
1,다른 사람이 나의 저작물을 패러디하였는데 저작권 침해에 해당하는 것일까요?,1.000,0.818182,0.38
2,사진이나 영상에 연예인의 유행어를 이용하는 것도 침해에 해당하는 것일까요?,1.000,0.500000,0.14
3,"이미 사망한 유명인을 모델로 하여 영화나 드라마를 제작하려고 하는데, 유족의 허락을...",0.875,1.000000,0.57
4,저작자가 명시적 또는 묵시적으로 동의한 범위 내에서 저작물을 변경할 경우 동일성유지...,0.800,1.000000,0.27
5,다른 사람의 저작물의 제호를 수정하여 사용할 경우 저작권 침해인가요?,1.000,0.000000,0.00
6,인터넷상에서 링크를 제공하는 것도 저작권법 위반에 해당하나요?,1.000,0.777778,0.35
7,저의 사진이 저작권 등록의 대상이 될 수 있을까요?,0.600,0.000000,0.00
8,"등록신청이 신청 서류의 하자 등으로 인하여 반려되었는데, 어떻게 하면 등록을 마무리...",1.000,0.000000,0.00
9,제가 등록신청한 저작물에 대한 등록관청의 심사 범위가 어디까지인가요?,1.000,0.000000,0.00


### RAGAS Metric Guide
- `llm_context_recall`: reference answer를 재현하는 데 필요한 정보가 retrieval context에 포함되어 있는지 평가
- `faithfulness`: 생성 답변이 retrieval context에 근거하고 있는지 평가
- `factual_correctness`: 생성 답변이 reference answer와 사실적으로 일치하는지 평가